# BillionMetaLab ASS 字幕特效 — 可视化分析

连接到 `ass_stats.db`，使用 matplotlib 绘制黑色主题量化图表。

**内核**: `conda activate mlx`  ·  **语言**: 简日/繁日已合并（日简→简日，日繁→繁日）

### Cell 1: Imports & 黑色主题 & 中文字体

In [ ]:
import sqlite3
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import matplotlib.ticker as mticker
import numpy as np
from pathlib import Path

# ── 黑色主题 ──
plt.style.use('dark_background')
# 微调：让背景更深、网格更暗
plt.rcParams.update({
    'figure.facecolor': '#0D0D0D',
    'axes.facecolor': '#161616',
    'axes.edgecolor': '#444444',
    'axes.labelcolor': '#DDDDDD',
    'text.color': '#DDDDDD',
    'xtick.color': '#AAAAAA',
    'ytick.color': '#AAAAAA',
    'grid.color': '#2A2A2A',
    'grid.alpha': 0.6,
    'legend.facecolor': '#1A1A1A',
    'legend.edgecolor': '#444444',
    'figure.dpi': 240,
    'savefig.dpi': 240,
    'savefig.bbox': 'tight',
    'savefig.facecolor': '#0D0D0D',
})

# ── 中文字体（强制重建缓存）──
fm._load_fontmanager(try_read_cache=False)
ZH_FONT = None
for name in ['Songti SC', 'STHeiti', 'Arial Unicode MS', 'PingFang HK', 'Kaiti SC']:
    for f in fm.fontManager.ttflist:
        if name == f.name:
            ZH_FONT = f
            break
    if ZH_FONT:
        break

if ZH_FONT:
    print(f'Font: {ZH_FONT.name}')
    plt.rcParams['font.family'] = ZH_FONT.name
else:
    print('WARNING: No Chinese font found')

plt.rcParams['axes.unicode_minus'] = False

# ── 数据库 ──
DB_PATH = Path('ass_stats.db').resolve()
conn = sqlite3.connect(str(DB_PATH))
conn.row_factory = sqlite3.Row

print(f'DB: {DB_PATH}')
print(f'Matplotlib: {matplotlib.__version__}')

### Cell 2: 辅助函数 & 色板

In [ ]:
def fetch(sql, params=None):
    cur = conn.cursor()
    return cur.execute(sql, params).fetchall() if params else cur.execute(sql).fetchall()


def label_bar(ax, fmt=':.2f', fontsize=7.5):
    """在条形上标注数值（自动判断水平/垂直）。"""
    for container in ax.containers:
        for rect in container:
            w, h = rect.get_width(), rect.get_height()
            val = w if w > h else h
            if val == 0:
                continue
            label = f'{val:{fmt}}' if fmt else (str(int(val)) if val == int(val) else f'{val:.2f}')
            if w > h:  # horizontal
                ax.annotate(label, xy=(w, rect.get_y() + h/2),
                            xytext=(4, 0), textcoords='offset points',
                            ha='left', va='center', fontsize=fontsize, color='#CCCCCC')
            else:      # vertical
                ax.annotate(label, xy=(rect.get_x() + w/2, h),
                            xytext=(0, 3), textcoords='offset points',
                            ha='center', va='bottom', fontsize=fontsize, color='#CCCCCC')


# 暗色主题优化色板（高饱和度，在黑色背景上凸显）
C = ['#5DADE2', '#F0B27A', '#58D68D', '#EC7063', '#AF7AC5',
     '#F7DC6F', '#85C1E9', '#BB8FCE', '#73C6B6', '#F1948A',
     '#7FB3D8', '#D7BDE2', '#A9DFBF', '#FAD7A0', '#AED6F1']

TIER_COLORS = ['#666666', '#58D68D', '#F4D03F', '#F0B27A', '#EC7063']
TIER_LABELS = ['极简', '简单', '中等', '复杂', '极复杂']

# 语言标签（已合并日简→简日、日繁→繁日）
LANG_CN = {
    'chs': '简体中文', 'cht': '繁体中文', 'jpn': '纯日文',
    'chs_jpn': '简日双语', 'cht_jpn': '繁日双语',
}

# Ready

### Chart 1: 季度特效复杂度趋势

In [ ]:
rows = fetch('''
    SELECT season, file_count,
           avg_complexity, avg_fx_per_dialogue, fx_dialogue_ratio
    FROM season_summary ORDER BY season
''')

seasons = [r['season'] for r in rows]
x = np.arange(len(seasons))

fig, ax1 = plt.subplots(figsize=(14, 5.5))

# 左轴：柱状 → 平均复杂度
bars = ax1.bar(x, [r['avg_complexity'] for r in rows],
               color=C[0], alpha=0.85, width=0.55, label='平均复杂度', zorder=3)
ax1.set_ylabel('平均复杂度 (0–10)', color=C[0], fontsize=11)
ax1.set_ylim(0, 11)
ax1.tick_params(axis='y', labelcolor=C[0])
ax1.grid(axis='y', alpha=0.3, zorder=0)

for bar, r in zip(bars, rows):
    h = bar.get_height()
    if h > 0:
        ax1.text(bar.get_x() + bar.get_width()/2, h + 0.15,
                 f'{r["avg_complexity"]:.2f}', ha='center', va='bottom', fontsize=8, color='#DDD')

# 右轴：折线 → 特效行占比
ax2 = ax1.twinx()
line = ax2.plot(x, [r['fx_dialogue_ratio'] * 100 for r in rows],
                color=C[3], marker='o', linewidth=2.2, markersize=7, label='特效行占比 (%)', zorder=4)
ax2.set_ylabel('特效行占 Dialogue 比 (%)', color=C[3], fontsize=11)
ax2.tick_params(axis='y', labelcolor=C[3])
ax2.set_ylim(0, 100)

for xi, r in zip(x, rows):
    pct = r['fx_dialogue_ratio'] * 100
    ax2.annotate(f'{pct:.1f}%', xy=(xi, pct), xytext=(0, 10),
                 textcoords='offset points', ha='center', fontsize=7, color=C[3])

ax1.set_xticks(x)
ax1.set_xticklabels(seasons, rotation=45, ha='right', fontsize=9)
ax1.set_xlabel('季度', fontsize=11)

l1 = plt.Line2D([0],[0], color=C[0], lw=6, alpha=0.85)
l2 = plt.Line2D([0],[0], color=C[3], marker='o', lw=2)
ax1.legend([l1, l2], ['平均复杂度', '特效行占比'], loc='upper left', fontsize=9)

ax1.set_title('季度特效复杂度 & 特效行占比趋势', fontsize=14, fontweight='bold', pad=15)
fig.tight_layout()
fig.savefig('chart1_quarterly_trend.png', dpi=240, facecolor='#0D0D0D')
plt.show()

### Chart 2: 特效类型使用率

In [ ]:
rows = fetch('''SELECT tag_category, files_using, usage_ratio, total_occurrences
               FROM effect_distribution ORDER BY usage_ratio DESC''')

CAT_CN = {
    'position': '定位 \\pos', 'color': '颜色 \\c', 'font_size': '字体大小 \\fs',
    'alpha': '透明度 \\alpha', 'border': '边框 \\bord', 'blur': '模糊 \\blur',
    'fade': '淡入淡出 \\fad', 'rotate': '旋转 \\fr', 'draw': '矢量绘图 \\p',
    'transform': '变换动画 \\t', 'move': '移动 \\move', 'font': '字体 \\fn',
    'spacing': '间距 \\fsp', 'clip': '裁剪遮罩 \\clip', 'karaoke': '卡拉OK \\k'
}

labels = [CAT_CN.get(r['tag_category'], r['tag_category']) for r in rows]
ratios = [r['usage_ratio'] * 100 for r in rows]
counts = [r['total_occurrences'] for r in rows]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# 左：使用率水平条形
bars1 = ax1.barh(range(len(labels)), ratios, color=C[:len(labels)], alpha=0.85, height=0.65, zorder=3)
ax1.set_yticks(range(len(labels)))
ax1.set_yticklabels(labels, fontsize=9)
ax1.invert_yaxis()
ax1.set_xlabel('使用率 (%)', fontsize=11)
ax1.set_title('特效类型使用率（文件维度）', fontsize=13, fontweight='bold')
ax1.set_xlim(0, 115)
ax1.grid(axis='x', alpha=0.3, zorder=0)

for bar, r, cnt in zip(bars1, ratios, counts):
    ax1.text(bar.get_width() + 1.5, bar.get_y() + bar.get_height()/2,
             f'{r:.1f}%  ({cnt:,})', va='center', fontsize=7.2, color='#BBB')

# 右：总出现次数 Top 8
idx = np.argsort(counts)
top8 = idx[-8:][::-1]
rest = sum(counts[i] for i in range(len(counts)) if i not in top8)
plot_lbl = [labels[i] for i in top8] + ['其他 7 类']
plot_val = [counts[i] for i in top8] + [rest]
plot_clr = C[:9]

bars2 = ax2.barh(range(len(plot_lbl)), plot_val, color=plot_clr, alpha=0.85, height=0.65, zorder=3)
ax2.set_yticks(range(len(plot_lbl)))
ax2.set_yticklabels(plot_lbl, fontsize=9)
ax2.invert_yaxis()
ax2.set_xlabel('总出现次数', fontsize=11)
ax2.set_title('特效标签出现次数 (Top 8)', fontsize=13, fontweight='bold')
ax2.grid(axis='x', alpha=0.3, zorder=0)

for bar, cnt in zip(bars2, plot_val):
    ax2.text(bar.get_width() + max(plot_val)*0.015, bar.get_y() + bar.get_height()/2,
             f'{cnt:,}', va='center', fontsize=8, color='#BBB')

fig.tight_layout()
fig.savefig('chart1_quarterly_trend.png', dpi=240, facecolor='#0D0D0D')
plt.show()

### Chart 3: 复杂度分档占比（环形图 + 条形）

In [ ]:
TIERS = [('极简', 0.0, 0.0), ('简单', 0.1, 2.0), ('中等', 2.1, 4.0), ('复杂', 4.1, 7.0), ('极复杂', 7.1, 10.0)]

tier_counts = []
for label, lo, hi in TIERS:
    if hi == 0.0:
        cnt = fetch('SELECT COUNT(*) AS n FROM effect_stats WHERE complexity_score = 0')[0]['n']
    else:
        cnt = fetch('''SELECT COUNT(*) AS n FROM effect_stats
                       WHERE complexity_score >= ? AND complexity_score <= ?''', (lo, hi))[0]['n']
    tier_counts.append(cnt)

total = sum(tier_counts)
tier_pcts = [round(c / total * 100, 2) for c in tier_counts]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))

# 左：环形图
wedges, texts, autotexts = ax1.pie(
    tier_counts, labels=TIER_LABELS, autopct='%1.2f%%',
    colors=TIER_COLORS, startangle=140, pctdistance=0.82,
    wedgeprops=dict(width=0.4, edgecolor='#0D0D0D', linewidth=2))
for at in autotexts:
    at.set_fontsize(9.5)
    at.set_fontweight('bold')
for t in texts:
    t.set_fontsize(10)
ax1.set_title('特效复杂度分档分布', fontsize=13, fontweight='bold')

# 右：水平条形
y = range(len(TIER_LABELS))[::-1]
cnts_r = tier_counts[::-1]
pcts_r = tier_pcts[::-1]
clrs_r = TIER_COLORS[::-1]

bars = ax2.barh(y, cnts_r, color=clrs_r, alpha=0.9, height=0.6, edgecolor='#222', linewidth=1, zorder=3)
ax2.set_yticks(y)
ax2.set_yticklabels(TIER_LABELS[::-1], fontsize=10)
ax2.set_xlabel('文件数', fontsize=11)
ax2.set_title('各档文件数量', fontsize=13, fontweight='bold')
ax2.grid(axis='x', alpha=0.3, zorder=0)

for bar, cnt, pct in zip(bars, cnts_r, pcts_r):
    ax2.text(bar.get_width() + max(cnts_r)*0.02, bar.get_y() + bar.get_height()/2,
             f'{cnt} 文件 ({pct:.2f}%)', va='center', fontsize=9, color='#CCC')
ax2.set_xlim(0, max(cnts_r) * 1.3)

fig.tight_layout()
fig.savefig('chart1_quarterly_trend.png', dpi=240, facecolor='#0D0D0D')
plt.show()

print('总计: %d 文件' % total)
for label, cnt, pct in zip(TIER_LABELS, tier_counts, tier_pcts):
    print('  %s: %d (%.2f%%)' % (label, cnt, pct))

### Chart 4: 高特效番剧 TOP 15

In [ ]:
rows = fetch('''
    SELECT f.anime_name, f.season, COUNT(*) AS file_count,
           ROUND(AVG(e.complexity_score), 2) AS avg_complexity,
           SUM(e.tag_draw) AS total_draw,
           SUM(e.tag_transform) AS total_transform,
           SUM(e.tag_karaoke) AS total_karaoke
    FROM file_stats f JOIN effect_stats e ON f.id = e.file_id
    GROUP BY f.season, f.anime_name
    ORDER BY avg_complexity DESC LIMIT 15
''')

labels = []
for r in rows:
    nm = r['anime_name']
    labels.append((nm[:11] + '…') if len(nm) > 12 else nm)

scores = [r['avg_complexity'] for r in rows]
draws  = [r['total_draw'] for r in rows]
trans  = [r['total_transform'] for r in rows]
karas  = [r['total_karaoke'] for r in rows]

fig, ax = plt.subplots(figsize=(13, 6.5))

y = range(len(labels))[::-1]
labels_r = labels[::-1]
scores_r = scores[::-1]

norm = plt.Normalize(0, 10)
bar_colors = plt.cm.YlOrRd(norm(scores_r))

bars = ax.barh(y, scores_r, color=bar_colors, alpha=0.9, height=0.65, edgecolor='#222', zorder=3)
ax.set_yticks(y)
ax.set_yticklabels(labels_r, fontsize=8.5)
ax.set_xlabel('平均复杂度 (0–10)', fontsize=11)
ax.set_title('高特效需求番剧 TOP 15（按平均复杂度）', fontsize=14, fontweight='bold')
ax.set_xlim(0, 10.5)
ax.grid(axis='x', alpha=0.3, zorder=0)

for bar, sc, dr, tr, kr in zip(bars, scores_r, draws[::-1], trans[::-1], karas[::-1]):
    parts = [f'{sc:.2f}']
    extras = []
    if dr > 0: extras.append(f'\p:{dr:,}')
    if tr > 0: extras.append(f'\t:{tr}')
    if kr > 0: extras.append(f'\k:{kr}')
    if extras:
        parts.append(' | ' + ' '.join(extras))
    ax.text(sc + 0.1, bar.get_y() + bar.get_height()/2,
            ''.join(parts), va='center', fontsize=7.2, color='#BBB')

fig.tight_layout()
fig.savefig('chart1_quarterly_trend.png', dpi=240, facecolor='#0D0D0D')
plt.show()

### Chart 5: 语言类型 × 特效交叉对比

In [ ]:
rows = fetch('''
    SELECT f.lang_type, COUNT(*) AS file_count,
           ROUND(AVG(e.complexity_score), 2)     AS avg_complexity,
           ROUND(AVG(e.total_effect_tags), 1)    AS avg_fx_tags,
           ROUND(AVG(CAST(e.dialogues_with_fx AS REAL) / MAX(f.dialogue_count, 1)) * 100, 1) AS fx_ratio,
           SUM(CASE WHEN e.tag_draw    > 0 THEN 1 ELSE 0 END) AS draw_files,
           SUM(CASE WHEN e.tag_karaoke > 0 THEN 1 ELSE 0 END) AS karaoke_files
    FROM file_stats f JOIN effect_stats e ON f.id = e.file_id
    GROUP BY f.lang_type ORDER BY file_count DESC
''')

labels = [LANG_CN.get(r['lang_type'], r['lang_type']) for r in rows]
avg_cx = [r['avg_complexity'] for r in rows]
avg_fx = [r['avg_fx_tags'] for r in rows]
fx_rat = [r['fx_ratio'] or 0 for r in rows]

x = np.arange(len(labels))
w = 0.22

fig, ax = plt.subplots(figsize=(12, 5.5))

b1 = ax.bar(x - w, avg_cx, w, color=C[0], alpha=0.9, label='平均复杂度 / 10', zorder=3)
b2 = ax.bar(x, avg_fx, w, color=C[1], alpha=0.9, label='平均特效标签数', zorder=3)
b3 = ax.bar(x + w, fx_rat, w, color=C[3], alpha=0.9, label='特效行占比 (%)', zorder=3)

ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=9)
ax.set_title('语言类型 × 特效复杂度交叉分析', fontsize=14, fontweight='bold')
ax.legend(loc='upper right', fontsize=8)
ax.grid(axis='y', alpha=0.3, zorder=0)

for bars in [b1, b2, b3]:
    for bar in bars:
        h = bar.get_height()
        ax.annotate(f'{h:.2f}' if h != int(h) else str(int(h)),
                     xy=(bar.get_x() + bar.get_width()/2, h),
                     xytext=(0, 3), textcoords='offset points',
                     ha='center', fontsize=6.5, color='#CCC')

fig.tight_layout()
fig.savefig('chart1_quarterly_trend.png', dpi=240, facecolor='#0D0D0D')
plt.show()

### Chart 6: 季度特效演变 — 绘图 & 卡拉OK

In [ ]:
rows = fetch('''
    SELECT season, file_count,
           files_with_draw, files_with_karaoke,
           ROUND(CAST(files_with_draw    AS REAL) / file_count * 100, 1) AS draw_pct,
           ROUND(CAST(files_with_karaoke AS REAL) / file_count * 100, 1) AS karaoke_pct,
           avg_complexity
    FROM season_summary ORDER BY season
''')

seasons = [r['season'] for r in rows]
x = np.arange(len(seasons))
draw_pct = [r['draw_pct'] for r in rows]
kara_pct = [r['karaoke_pct'] for r in rows]
avg_cx   = [r['avg_complexity'] for r in rows]

fig, ax1 = plt.subplots(figsize=(14, 5.5))

# 堆叠柱状
b1 = ax1.bar(x, draw_pct, color=C[3], alpha=0.85, label='含 \\p 绘图 (%)', width=0.55, zorder=3)
b2 = ax1.bar(x, kara_pct, bottom=draw_pct, color=C[0], alpha=0.85, label='含 \\k 卡拉OK (%)', width=0.55, zorder=3)

ax1.set_ylabel('文件占比 (%)', fontsize=11)
top_lim = max(d + k for d, k in zip(draw_pct, kara_pct))
ax1.set_ylim(0, top_lim * 1.5 + 8)
ax1.set_xticks(x)
ax1.set_xticklabels(seasons, rotation=45, ha='right', fontsize=9)
ax1.grid(axis='y', alpha=0.3, zorder=0)

for i in range(len(x)):
    if draw_pct[i] > 0:
        ax1.text(x[i], draw_pct[i]/2, f'{draw_pct[i]:.1f}%', ha='center', fontsize=7, color='#0D0D0D', fontweight='bold')
    if kara_pct[i] > 0:
        ax1.text(x[i], draw_pct[i] + kara_pct[i]/2, f'{kara_pct[i]:.1f}%', ha='center', fontsize=7, color='white', fontweight='bold')

# 右轴：复杂度折线
ax2 = ax1.twinx()
ax2.plot(x, avg_cx, color='#F9E79F', marker='s', linewidth=2.2, markersize=7, label='平均复杂度', zorder=4)
ax2.set_ylabel('平均复杂度 (0–10)', color='#F9E79F', fontsize=11)
ax2.tick_params(axis='y', labelcolor='#F9E79F')
ax2.set_ylim(0, 11)

for i in range(len(x)):
    ax2.annotate(f'{avg_cx[i]:.2f}', (x[i], avg_cx[i]),
                 xytext=(0, 10), textcoords='offset points', ha='center', fontsize=7.5, color='#F9E79F')

h1, l1 = ax1.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(h1 + h2, l1 + l2, loc='upper left', fontsize=8)

ax1.set_xlabel('季度', fontsize=11)
plt.title('季度特效演变：\p 绘图 & \k 卡拉OK 使用率 + 复杂度', fontsize=14, fontweight='bold')
fig.tight_layout()
fig.savefig('chart1_quarterly_trend.png', dpi=240, facecolor='#0D0D0D')
plt.show()

### Chart 7: 特效标签使用矩阵（季度 × 类型热力图）

In [ ]:
raw = fetch('''
    SELECT f.season,
           SUM(e.tag_draw) AS draw, SUM(e.tag_transform) AS transform,
           SUM(e.tag_karaoke) AS karaoke, SUM(e.tag_move) AS move,
           SUM(e.tag_clip) AS clip, SUM(e.tag_fade) AS fade
    FROM file_stats f JOIN effect_stats e ON f.id = e.file_id
    GROUP BY f.season ORDER BY f.season
''')

CATS = ['draw', 'transform', 'move', 'clip', 'karaoke', 'fade']
CAT_CN_MAT = {'draw': '矢量绘图', 'transform': '变换动画', 'move': '移动',
              'clip': '裁剪遮罩', 'karaoke': '卡拉OK', 'fade': '淡入淡出'}

seasons = [r['season'] for r in raw]
data = np.array([[r[cat] for cat in CATS] for r in raw], dtype=float)

# 归一化
data_norm = np.zeros_like(data)
for j in range(len(CATS)):
    col = data[:, j]
    if col.max() > 0:
        data_norm[:, j] = col / col.max()

fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(data_norm.T, cmap='inferno', aspect='auto', vmin=0, vmax=1)

ax.set_xticks(range(len(seasons)))
ax.set_xticklabels(seasons, rotation=45, ha='right', fontsize=9)
ax.set_yticks(range(len(CATS)))
ax.set_yticklabels([CAT_CN_MAT[c] for c in CATS], fontsize=10)
ax.set_title('季度 × 高成本特效标签 热力图（归一化）', fontsize=14, fontweight='bold')

for i in range(len(seasons)):
    for j in range(len(CATS)):
        val = data[i, j]
        if val > 0:
            clr = '#0D0D0D' if data_norm[i, j] > 0.55 else '#EEE'
            ax.text(i, j, f'{int(val):,}', ha='center', va='center', fontsize=7.5, color=clr, fontweight='bold')

cbar = fig.colorbar(im, ax=ax, shrink=0.85, pad=0.02)
cbar.set_label('归一化强度', fontsize=10)
cbar.ax.yaxis.set_tick_params(color='#AAA')

fig.tight_layout()
fig.savefig('chart1_quarterly_trend.png', dpi=240, facecolor='#0D0D0D')
plt.show()

### Chart 8: 季度文件 & 番剧规模概览

In [ ]:
rows = fetch('''SELECT season, file_count, anime_count, total_lines, avg_styles
               FROM season_summary ORDER BY season''')

seasons = [r['season'] for r in rows]
files   = [r['file_count'] for r in rows]
animes  = [r['anime_count'] for r in rows]
lines_k = [r['total_lines'] / 1000 for r in rows]

x = np.arange(len(seasons))
w = 0.26

fig, ax = plt.subplots(figsize=(14, 5.5))

b1 = ax.bar(x - w, files, w, color=C[0], alpha=0.9, label='ASS 文件数', zorder=3)
b2 = ax.bar(x, animes, w, color=C[2], alpha=0.9, label='番剧数', zorder=3)
b3 = ax.bar(x + w, lines_k, w, color=C[4], alpha=0.9, label='总行数 (千行)', zorder=3)

ax.set_xticks(x)
ax.set_xticklabels(seasons, rotation=45, ha='right', fontsize=9)
ax.set_title('各季度文件数 / 番剧数 / 总规模', fontsize=14, fontweight='bold')
ax.legend(loc='upper left', fontsize=8)
ax.grid(axis='y', alpha=0.3, zorder=0)

for bars in [b1, b2, b3]:
    for bar in bars:
        h = bar.get_height()
        if h > 0:
            ax.annotate(f'{h:.0f}' if h == int(h) else f'{h:.1f}',
                         xy=(bar.get_x() + bar.get_width()/2, h),
                         xytext=(0, 3), textcoords='offset points',
                         ha='center', fontsize=6.2, color='#CCC')

fig.tight_layout()
fig.savefig('chart1_quarterly_trend.png', dpi=240, facecolor='#0D0D0D')
plt.show()

### Chart 9: 特效使用密度 TOP 15（气泡图）

In [ ]:
rows = fetch('''
    SELECT f.anime_name, f.season,
           ROUND(AVG(CAST(e.total_effect_tags AS REAL) / MAX(f.dialogue_count, 1)), 3) AS fx_density,
           ROUND(AVG(e.complexity_score), 2) AS avg_complexity
    FROM file_stats f JOIN effect_stats e ON f.id = e.file_id
    GROUP BY f.season, f.anime_name
    ORDER BY fx_density DESC LIMIT 15
''')

labels = []
for r in rows:
    nm = r['anime_name']
    labels.append((nm[:13] + '…') if len(nm) > 14 else nm)

density = [r['fx_density'] for r in rows]
complexity = [r['avg_complexity'] for r in rows]

fig, ax = plt.subplots(figsize=(12, 5.5))

y_pos = range(len(labels))[::-1]
dx_r = density[::-1]
cx_r = complexity[::-1]
lbl_r = labels[::-1]

sizes = [max(c * 100, 35) for c in cx_r]
colors = plt.cm.YlOrRd([c/10 for c in cx_r])

ax.scatter(dx_r, y_pos, s=sizes, c=colors, alpha=0.9, edgecolors='#444', linewidth=0.8, zorder=5)
ax.set_yticks(y_pos)
ax.set_yticklabels(lbl_r, fontsize=8.5)
ax.set_xlabel('每 Dialogue 平均特效标签数', fontsize=11)
ax.set_title('特效使用密度 TOP 15（气泡大小 = 复杂度）', fontsize=14, fontweight='bold')
ax.grid(axis='x', alpha=0.3, zorder=0)

for i, (dx, cx) in enumerate(zip(dx_r, cx_r)):
    ax.annotate(f'{dx:.3f}', (dx, i), xytext=(7, 0), textcoords='offset points',
                ha='left', va='center', fontsize=7, color='#BBB')

ax.set_xlim(0, max(dx_r) * 1.25)
fig.tight_layout()
fig.savefig('chart1_quarterly_trend.png', dpi=240, facecolor='#0D0D0D')
plt.show()

---

## 全局快照

In [ ]:
cur = conn.cursor()

n_files   = cur.execute('SELECT COUNT(*) FROM file_stats').fetchone()[0]
n_lines   = cur.execute('SELECT SUM(total_lines) FROM file_stats').fetchone()[0]
n_dialog  = cur.execute('SELECT SUM(dialogue_count) FROM file_stats').fetchone()[0]
n_anime   = cur.execute('SELECT COUNT(DISTINCT anime_name) FROM file_stats').fetchone()[0]
n_season  = cur.execute('SELECT COUNT(DISTINCT season) FROM file_stats').fetchone()[0]
avg_compl = cur.execute('SELECT ROUND(AVG(complexity_score), 2) FROM effect_stats').fetchone()[0]
zero_fx   = cur.execute('SELECT COUNT(*) FROM effect_stats WHERE complexity_score = 0').fetchone()[0]
draw_f    = cur.execute('SELECT COUNT(*) FROM effect_stats WHERE tag_draw > 0').fetchone()[0]
kara_f    = cur.execute('SELECT COUNT(*) FROM effect_stats WHERE tag_karaoke > 0').fetchone()[0]

print('=' * 55)
print('  BillionMetaLab ASS 字幕合集 — 量化总览')
print('=' * 55)
print(f'  文件总数:      {n_files}')
print(f'  总行数:        {n_lines:,}')
print(f'  总 Dialogue:   {n_dialog:,}')
print(f'  覆盖番剧:      {n_anime} 部')
print(f'  覆盖季度:      {n_season} 个')
print(f'  平均复杂度:    {avg_compl} / 10')
print(f'  零特效文件:    {zero_fx} ({zero_fx/n_files*100:.2f}%)')
print(f'  含矢量绘图:    {draw_f} ({draw_f/n_files*100:.2f}%)')
print(f'  含卡拉OK:      {kara_f} ({kara_f/n_files*100:.2f}%)')
print('=' * 55)

# 语言分布
print('\n  语言分布:')
for row in cur.execute('SELECT lang_type, COUNT(*) FROM file_stats GROUP BY lang_type ORDER BY COUNT(*) DESC'):
    label = LANG_CN.get(row[0], row[0])
    print(f'    {label}: {row[1]}')

conn.close()
print('\nConnection closed.')

---

## 完成

以上 9 张图表覆盖：

| 图表 | 说明 |
|------|------|
| Chart 1–2 | 趋势 & 分布概览 |
| Chart 3 | 复杂度分档占比 |
| Chart 4 | 高特效番剧 TOP 15 |
| Chart 5 | 语言类型交叉分析 |
| Chart 6–7 | 季度演变 & 热力图 |
| Chart 8–9 | 规模 & 密度 |

所有图表采用 **暗色主题**，语言标签已合并（日简→简日、日繁→繁日）。